# Remove selected cases from MongoDB

This notebook resolves the listed neutral citations to their source judgement records and removes the matching documents from the related collections.

Run the resolution cell first, inspect the matched IDs, then run the deletion cell.

- `[2023] HKDC 748`
- `[2022] HKDC 1522`
- `[2022] HKDC 565`
- `[2021] HKDC 453`


In [4]:
import os
import re
from pathlib import Path
from pprint import pprint

from bson import ObjectId
from dotenv import load_dotenv
from pymongo import MongoClient

for dotenv_path in (
    Path.cwd() / '.env',
    Path.cwd() / 'notebooks' / '.env',
    Path.cwd().parent / 'notebooks' / '.env',
):
    if dotenv_path.exists():
        load_dotenv(dotenv_path)
        break

uri = os.getenv('DB_MONGODB_URI')
if not uri:
    raise RuntimeError('DB_MONGODB_URI is not set')

client = MongoClient(uri)
db = client['drug-sentencing-predictor']
judgements = db['judgement-html']
extracted = db['llm-extracted-features']
verified = db['verified-features']
locks = db['verification-locks']

cases = [
    '[2023] HKDC 748',
    '[2022] HKDC 1522',
    '[2022] HKDC 565',
    '[2021] HKDC 453',
    '[2025] HKDC 714',
    '[2024] HKDC 1239'
]

citation_re = re.compile(r'^\[(\d{4})\]\s+([A-Z]+)\s+(\d+)$')


def parse_citation(citation):
    match = citation_re.fullmatch(citation)
    if not match:
        raise ValueError(f'Unsupported citation format: {citation}')
    return match.groups()


def to_object_id(value):
    if isinstance(value, ObjectId):
        return value
    return ObjectId(str(value))


def resolve_source_ids(citation):
    source_ids = set()

    for doc in verified.find(
        {'judgement.neutral_citation': citation},
        {'source_judgement_id': 1},
    ):
        source_judgement_id = doc.get('source_judgement_id')
        if source_judgement_id is not None:
            source_ids.add(to_object_id(source_judgement_id))

    if source_ids:
        return source_ids

    year, _, trial = parse_citation(citation)
    for doc in judgements.find(
        {'year': year, 'trial': trial, 'appeal': None, 'corrigendum': None},
        {'_id': 1, 'filename': 1, 'trial': 1, 'year': 1, 'appeal': 1, 'corrigendum': 1},
    ):
        source_ids.add(to_object_id(doc['_id']))

    return source_ids


In [5]:
source_ids_by_case = {citation: resolve_source_ids(citation) for citation in cases}

for citation, source_ids in source_ids_by_case.items():
    print(citation)
    if not source_ids:
        print('  no matching source judgement found')
        continue

    for source_id in sorted(source_ids, key=str):
        doc = judgements.find_one(
            {'_id': source_id},
            {'filename': 1, 'trial': 1, 'year': 1, 'appeal': 1, 'corrigendum': 1},
        )
        print(' ', source_id, doc)

target_source_ids = sorted(
    {source_id for source_ids in source_ids_by_case.values() for source_id in source_ids},
    key=str,
)

if not target_source_ids:
    raise RuntimeError('No matching source judgement IDs were resolved')

print()
print('unique source judgement IDs:', len(target_source_ids))
print('related documents to delete:')
print('  verification-locks:', locks.count_documents({'source_judgement_id': {'$in': target_source_ids}}))
print('  verified-features:', verified.count_documents({'source_judgement_id': {'$in': target_source_ids}}))
print('  llm-extracted-features:', extracted.count_documents({'source_judgement_id': {'$in': target_source_ids}}))
print('  judgement-html:', judgements.count_documents({'_id': {'$in': target_source_ids}}))


[2023] HKDC 748
  69b2f271296bb506fb4888c0 {'_id': ObjectId('69b2f271296bb506fb4888c0'), 'filename': '[2023] HKDC 748.htm', 'year': '2023', 'trial': '[2023] HKDC 748', 'appeal': None, 'corrigendum': None}
[2022] HKDC 1522
  69b2f26f296bb506fb4886f5 {'_id': ObjectId('69b2f26f296bb506fb4886f5'), 'filename': '[2022] HKDC 1522.htm', 'year': '2022', 'trial': '[2022] HKDC 1522', 'appeal': None, 'corrigendum': None}
[2022] HKDC 565
  69b2f26f296bb506fb4886b4 {'_id': ObjectId('69b2f26f296bb506fb4886b4'), 'filename': '[2022] HKDC 565.htm', 'year': '2022', 'trial': '[2022] HKDC 565', 'appeal': None, 'corrigendum': None}
[2021] HKDC 453
  69b2f26d296bb506fb4885f4 {'_id': ObjectId('69b2f26d296bb506fb4885f4'), 'filename': '[2021] HKDC 453.htm', 'year': '2021', 'trial': '[2021] HKDC 453', 'appeal': None, 'corrigendum': None}
[2025] HKDC 714
  69b2f273296bb506fb488d9c {'_id': ObjectId('69b2f273296bb506fb488d9c'), 'filename': '[2025] HKDC 714_C1.htm', 'year': '2025', 'trial': '[2025] HKDC 714', 'appea

In [6]:
delete_results = {}

delete_results['verification-locks'] = locks.delete_many(
    {'source_judgement_id': {'$in': target_source_ids}}
).deleted_count
delete_results['verified-features'] = verified.delete_many(
    {'source_judgement_id': {'$in': target_source_ids}}
).deleted_count
delete_results['llm-extracted-features'] = extracted.delete_many(
    {'source_judgement_id': {'$in': target_source_ids}}
).deleted_count
delete_results['judgement-html'] = judgements.delete_many(
    {'_id': {'$in': target_source_ids}}
).deleted_count

pprint(delete_results)


{'judgement-html': 6,
 'llm-extracted-features': 6,
 'verification-locks': 0,
 'verified-features': 6}


In [7]:
print('verification-locks remaining:', locks.count_documents({'source_judgement_id': {'$in': target_source_ids}}))
print('verified-features remaining:', verified.count_documents({'source_judgement_id': {'$in': target_source_ids}}))
print('llm-extracted-features remaining:', extracted.count_documents({'source_judgement_id': {'$in': target_source_ids}}))
print('judgement-html remaining:', judgements.count_documents({'_id': {'$in': target_source_ids}}))


verification-locks remaining: 0
verified-features remaining: 0
llm-extracted-features remaining: 0
judgement-html remaining: 0
